# Task B — From R to Python: Translating Statistical Analysis
> **BAFAD Accelerated Research Track · Fall 2026**

This task bridges your STAT 270 R work with the Python data science ecosystem used in AI research.
You will reproduce three analyses from Assignment 1 — originally written in R using `dplyr` and `ggplot2` — in Python using `pandas` and `matplotlib`.

**Reference file:** `task_b_r_original.R` (in this folder) — contains the complete R code you are translating.

**Data:** `../data/flights.csv` and `../data/airports.csv`

**Submission:** Push your completed notebook to your fork of the starter repository and paste the repo URL into the Canvas assignment.

## Setup

Run this cell first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NAVY = '#021B3A'
BLUE = '#0EA5E9'
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})

flights  = pd.read_csv('../data/flights.csv')
airports = pd.read_csv('../data/airports.csv')

print(f'flights:  {flights.shape[0]:,} rows × {flights.shape[1]} columns')
print(f'airports: {airports.shape[0]} rows × {airports.shape[1]} columns')
flights.head(3)

---
## Question 2 — Frequency Tables and Proportions (10 pts)

### Part A (4 pts)
Compute the **number of flights** and the **proportion of all flights** operated by each carrier.
Display the result sorted from most to least flights.

**R equivalent:**
```r
flights |> count(carrier, sort = TRUE) |> mutate(prop = n / sum(n))
```

In [ ]:
counts = flights['carrier'].value_counts()
carrier_counts = pd.DataFrame({
    'carrier': counts.index,
    'n': counts.values,
    'prop': counts.values / len(flights)
})

carrier_counts

**Answer** — Which carrier operated the most flights? What was its approximate share?

> United Airlines (`UA`) operated the most flights, accounting for about 17% of all flights.

### Part B (6 pts)
Compute the **proportion of flights that arrived late** (`arr_delay > 0`) for each carrier,
sorted from highest to lowest late-arrival rate.

In [ ]:
late_arrivals = (
    flights.dropna(subset=['arr_delay'])
    .groupby('carrier')['arr_delay']
    .apply(lambda x: (x > 0).mean())
    .reset_index(name='late_rate')
    .sort_values('late_rate', ascending=False)
    .reset_index(drop=True)
)

late_arrivals

**Answer** — Which carrier had the worst on-time arrival record?

> Frontier Airlines (`F9`) had the highest proportion of late arrivals, so it had the worst on-time arrival record.

---
## Question 3 — Histogram, Density, and Shape (15 pts)

### Part A (7 pts)
Create a **density histogram** of `dep_delay`, restricted to flights where `dep_delay` is between −60 and 180 minutes.
Overlay a kernel density estimate (KDE) curve.

In [ ]:
from scipy.stats import gaussian_kde

df_filtered = flights[(flights['dep_delay'] >= -60) & (flights['dep_delay'] <= 180)].copy()
delay = df_filtered['dep_delay'].dropna()

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(delay, bins=30, density=True, color=NAVY, alpha=0.75, edgecolor='white')
kde = gaussian_kde(delay)
x_vals = np.linspace(delay.min(), delay.max(), 500)
ax.plot(x_vals, kde(x_vals), color=BLUE, linewidth=2)

ax.set_xlabel('Departure Delay (minutes)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Departure Delays (−60 to 180 min)')
plt.tight_layout()
plt.show()

### Part B (8 pts)
Compute and report the **median** and **IQR** of `dep_delay` (same filtered range), then describe the distribution in complete sentences addressing: shape, modality, centre, and spread.

In [ ]:
median_delay = delay.median()
iqr_delay = delay.quantile(0.75) - delay.quantile(0.25)
print(f'Median departure delay: {median_delay:.1f} min')
print(f'IQR of departure delay: {iqr_delay:.1f} min')

**Answer** — Shape, modality, centre, spread:

> The departure-delay distribution is unimodal and strongly right-skewed. Most flights leave close to their scheduled departure time, while a smaller number have much larger positive delays. The median is close to zero, and the IQR shows that the middle half of flights is concentrated in a much narrower range than the long right tail.

---
## Question 4 — Grouped Summaries by Origin Airport (15 pts)

### Part A (7 pts)
For each of the three origin airports (EWR, JFK, LGA), compute the **mean, median, standard deviation, and IQR** of `arr_delay`.
Display the results as a summary table.

In [ ]:
from scipy.stats import iqr as scipy_iqr

origin_summary = (
    flights.groupby('origin')['arr_delay']
    .agg(
        mean_arr='mean',
        median_arr='median',
        sd_arr='std',
        iqr_arr=lambda x: x.quantile(0.75) - x.quantile(0.25)
    )
    .reset_index()
)

origin_summary

### Part B (8 pts)
Based on your table, answer in complete sentences:
1. Which airport has the **highest mean** arrival delay? Lowest?
2. For each airport, is the mean **higher or lower** than the median? What does this imply about the shape of each delay distribution?
3. Which airport shows the **greatest spread** (use SD to support your answer)?

**Answer:**

> JFK has the highest mean arrival delay, while LGA has the lowest. For EWR, JFK, and LGA, the mean is higher than the median, which indicates that all three arrival-delay distributions are right-skewed because large positive delays pull the mean upward. JFK also has the greatest spread because it has the largest standard deviation.

---
## Reflection (not graded — but read before Task C)

The hardest part of translating from R to Python was matching `dplyr` grouped operations to the correct `pandas` syntax. I found `groupby()` useful because it can create the same types of summaries as R while keeping everything in one table. One difference is that `dplyr` often reads like a sequence of pipeline steps, while `pandas` uses method chaining and indexing more heavily.